In [ ]:
import tensorflow as tf

In [ ]:
import tensorflow_datasets as tfds

In [ ]:
training_data = tfds.load('cats_vs_dogs',split="train[:80%]")
testing_data = tfds.load('cats_vs_dogs',split="train[80%:]")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.HZMYOM_4.0.1/cats_vs_dogs-train.tfrecord-[0-9][0-9…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.


In [ ]:
x_train = training_data.element_spec['image']
y_train = training_data.element_spec['label']
x_test = testing_data.element_spec['image']
y_test = testing_data.element_spec['label']

In [ ]:

print('x_train shape : ',x_train.shape)
print('y_train shape : ',y_train.shape)
print('x_test shape : ',x_test.shape)
print('y_test shape : ',y_test.shape)

x_train shape :  (None, None, 3)
y_train shape :  ()
x_test shape :  (None, None, 3)
y_test shape :  ()


In [ ]:
def preprocess_image(element):
    print(element)
    image = element['image']
    label = element['label']

    resized_image = tf.image.resize(image, (160, 160)) / 255.0
    return resized_image, label

train_dataset = training_data.map(preprocess_image)
test_dataset = testing_data.map(preprocess_image)
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

{'image': <tf.Tensor 'args_0:0' shape=(None, None, 3) dtype=uint8>, 'image/filename': <tf.Tensor 'args_1:0' shape=() dtype=string>, 'label': <tf.Tensor 'args_2:0' shape=() dtype=int64>}
{'image': <tf.Tensor 'args_0:0' shape=(None, None, 3) dtype=uint8>, 'image/filename': <tf.Tensor 'args_1:0' shape=() dtype=string>, 'label': <tf.Tensor 'args_2:0' shape=() dtype=int64>}


In [ ]:
from tensorflow.keras import models, layers
from tensorflow.keras.layers import Dropout
import tensorflow as tf

In [ ]:
data_augmentation = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = models.Sequential()
model.add(data_augmentation)

model.add(layers.Conv2D(8,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))
model.add(Dropout(0.2))

model.add(layers.Conv2D(16,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))
model.add(Dropout(0.2))

model.add(layers.Conv2D(32,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))
model.add(Dropout(0.2))

model.add(layers.Conv2D(64,(3,3),activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(64,activation='relu'))
model.add(layers.Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 158, 158, 8)    │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 158, 158, 8)    │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 79, 79, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 79, 79, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 77, 77, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 77, 77, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 38, 38, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 38, 38, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 36, 36, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 36, 36, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 18, 18, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,233 (114.19 KB)

 Trainable params: 28,993 (113.25 KB)

 Non-trainable params: 240 (960.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=0.00001
)

history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    callbacks=[lr_callback]
)

Epoch 1/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 542s 923ms/step - accuracy: 0.6416 - loss: 0.6323 - val_accuracy: 0.5963 - val_loss: 0.6969 - learning_rate: 0.0010
Epoch 2/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 529s 909ms/step - accuracy: 0.6991 - loss: 0.5779 - val_accuracy: 0.7276 - val_loss: 0.5509 - learning_rate: 0.0010
Epoch 3/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 525s 902ms/step - accuracy: 0.7259 - loss: 0.5415 - val_accuracy: 0.7524 - val_loss: 0.5275 - learning_rate: 0.0010
Epoch 4/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 561s 900ms/step - accuracy: 0.7451 - loss: 0.5157 - val_accuracy: 0.7631 - val_loss: 0.5182 - learning_rate: 0.0010
Epoch 5/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 531s 911ms/step - accuracy: 0.7581 - loss: 0.4942 - val_accuracy: 0.6948 - val_loss: 0.6691 - learning_rate: 0.0010
Epoch 6/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 524s 900ms/step - accuracy: 0.7753 - loss: 0.4692 - val_accuracy: 0.7171 - val_loss: 0.5941 - learning_rate: 0.0010
Epoch 7/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 516s 886ms/step - accura

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = 'sample.jpg'

img = image.load_img(img_path, target_size=(160, 160))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print(f"Prediction: DOG ({prediction[0][0]*100:.2f}%)")
else:
    print(f"Prediction: CAT ({(1 - prediction[0][0])*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step
Prediction: CAT (98.03%)


Training the same Dataset with Transfer Learning.


In [ ]:
from tensorflow.keras.applications import MobileNetV2

In [ ]:
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model_2 = models.Sequential()
model_2.add(base_model)
model_2.add(layers.GlobalAveragePooling2D())
model_2.add(layers.Dense(64, activation='relu'))
model_2.add(layers.Dropout(0.3))
model_2.add(layers.Dense(1, activation='sigmoid'))

In [ ]:
model_2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history_2 = model_2.fit(train_dataset,epochs=5,validation_data=test_dataset)

Epoch 1/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 472s 800ms/step - accuracy: 0.9668 - loss: 0.0839 - val_accuracy: 0.9779 - val_loss: 0.0575
Epoch 2/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 469s 805ms/step - accuracy: 0.9787 - loss: 0.0552 - val_accuracy: 0.9800 - val_loss: 0.0565
Epoch 3/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 516s 887ms/step - accuracy: 0.9843 - loss: 0.0452 - val_accuracy: 0.9800 - val_loss: 0.0585
Epoch 4/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 468s 804ms/step - accuracy: 0.9853 - loss: 0.0398 - val_accuracy: 0.9809 - val_loss: 0.0593
Epoch 5/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 519s 891ms/step - accuracy: 0.9886 - loss: 0.0320 - val_accuracy: 0.9798 - val_loss: 0.0639


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = 'sample.jpg'

img = image.load_img(img_path, target_size=(160, 160))
img_array = img / 255.0

prediction = model_2.predict(img_array)

if prediction[0][0] > 0.5:
    print(f"Prediction: DOG ({prediction[0][0]*100:.2f}%)")
else:
    print(f"Prediction: CAT ({(1 - prediction[0][0])*100:.2f}%)")